In [1]:
import pandas as pd
from datasets import Dataset
df = pd.read_csv('/home/wagyu0923/project/Document_Analyzer/evaluation_data.csv')


/home/wagyu0923/miniconda3/envs/exaone/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
import sys
import os
sys.path.append('/home/wagyu0923/project/Document_Analyzer')
from pipeline.document_loader import DocumentLoader
from pipeline.chunker import Chunker
from pipeline.embedder import Embedder
from pipeline.vector_retriever import VectorRetriever
from pipeline.generator import Generator
import config
from tkinter import filedialog

def setup_pipeline():
    chunker = Chunker(
        chunk_size = config.CHUNK_SIZE,
        overlap_size = config.OVERLAP_SIZE
    )
    print('Chunking Complete')
    embedder = Embedder(
        model_name = config.EMBEDDING_MODEL
    )
    print('Embedding Complete')
    retriever =VectorRetriever(
        db_path = config.DB_PATH,
        model_name = config.EMBEDDING_MODEL,
        collection_name = config.COLLECTION_NAME
    )
    print('Retrieving Coplete')
    generator = Generator(
        model_name = config.LLM_NAME,
        options = config.DEFAULT_OLLAMA_OPTIONS
    )
    return chunker, embedder, retriever, generator

def run_indexing(file_path, chunker, embedder, retriever):
    loader = DocumentLoader(file_path = file_path)
    document = loader.load()
    chunks = chunker.chunking(document)
    embedded_chunks = embedder.embed_documents(chunks)
    file_name = os.path.basename(file_path)
    retriever.add_documents(embedded_chunks, file_name)

chunker, embedder, retriever, generator = setup_pipeline()

file_path = '/home/wagyu0923/project/Document_Analyzer/pdf_files/[세토피아][정정]반기보고서(2025.09.09).pdf'
run_indexing(file_path, chunker, embedder, retriever)

Chunking Complete
Embedding Complete
Retrieving Coplete


In [4]:
import json
dataset = df.copy()
for index, query in enumerate(df['user_input']):
    retrieved_data = retriever.retrieve(query, n_results = 5)
    outputs = generator.generate(retrieved_data, query)
    try:
        outputs = json.loads(outputs)
    except json.JSONDecodeError:
        print(f'JSON Decode Error at index {index+1}. Skipping.') 
        print(outputs)
        continue 
    if 'used_context' not in outputs.keys():
        outputs['used_context'] = []
    elif 'answer' not in outputs.keys():
        outputs['answer'] = ''
    dataset.loc[index, 'retrieved_contexts'] = outputs['used_context']
    dataset.loc[index, 'response'] = outputs['answer']
    print(f'progress : {index+1}/{len(df)}')
    print(outputs)




/tmp/ipykernel_2673/3642859904.py:16: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '【 대표이사 등의 확인 】 전자공시시스템 dart.fss.or.kr Page 3반 기 보 고 서 (제 11 기) 사업연도 2025년 01월 01일 부터 2025년 06월 30일 까지 금융위원회 한국거래소 귀중 2025년 08월 14일 제출대상법인 유형 : 주권상장법인 면제사유발생 : 해당사항 없음 회 사 명 : (주)세토피아 대 표 이 사 : 서상철 본 점 소 재 지 : 서울시 강남구 삼성로 81길35, 3층 (전 화) 02-3497-8900 (홈페이지) http://www.se-topia.com/ 작 성 책 임 자 :  ' has dtype incompatible with float64, please explicitly cast to a compatible dtype first.
  dataset.loc[index, 'retrieved_contexts'] = outputs['used_context']
/tmp/ipykernel_2673/3642859904.py:17: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '서상철' has dtype incompatible with float64, please explicitly cast to a compatible dtype first.
  dataset.loc[index, 'response'] = outputs['answer']


progress : 1/30
{'answer': '서상철', 'used_context': '【 대표이사 등의 확인 】 전자공시시스템 dart.fss.or.kr Page 3반 기 보 고 서 (제 11 기) 사업연도 2025년 01월 01일 부터 2025년 06월 30일 까지 금융위원회 한국거래소 귀중 2025년 08월 14일 제출대상법인 유형 : 주권상장법인 면제사유발생 : 해당사항 없음 회 사 명 : (주)세토피아 대 표 이 사 : 서상철 본 점 소 재 지 : 서울시 강남구 삼성로 81길35, 3층 (전 화) 02-3497-8900 (홈페이지) http://www.se-topia.com/ 작 성 책 임 자 :  '}
progress : 2/30
{'answer': '(주)마이더스AI', 'used_context': '2022년 8월 1일 임시주주총회 이후 (주)마이더스AI에서 (주)세토피아로 상호명이 변경되었 습니다.'}
progress : 3/30
{'answer': '2023.01.02', 'used_context': '합병기일 2023.01.02'}
progress : 4/30
{'answer': '제공된 컨텍스트만으로는 질문에 답변할 수 없습니다.', 'used_context': '<철강사업> 당사는 철강, 스테인레스강, 특수강 등의 도매, 제조사업을 영위하고 있으며 STS 316 430 201 등의 강종을 취급하고 있습니다.  당사는 국내 건설, 기계 자동차 산업분야의 약 300여개의 거래처와 상시 거래하고 있으며, 국내외 다양한 매입처로부터 스테인리스강을 확보하여 매월 고정적인 수요에 맞추어 주문, 생산, 판매하고 있습니다. 판매경로는 부천공장, 시화 하치장에서 고객사에게 직접 출고되며, 외부 임가공처에서 가공 후에 고객사에 출고되는 경우가 있습니다.'}
progress : 5/30
{'answer': 'ELFBAR(엘프바)', 'used_context': '2월 글로벌 1위 전자담배브랜드 ELFBAR(엘프바) 제품 국내 총판 독점 계약을 체결 하여

In [6]:
import ast  # ast 모듈을 임포트합니다.
import json # (기존 코드)
import pandas as pd # (기존 코드)

# 'json.loads(x)'를 'ast.literal_eval(x)'로 변경
dataset["retrieved_contexts"] = dataset["retrieved_contexts"].apply(
    lambda x: []
    if x is None or (isinstance(x, float) and pd.isna(x)) or x == ""
    # 이 부분을 수정합니다.
    else (ast.literal_eval(x.strip()) if isinstance(x, str) and x.strip().startswith("[") and x.strip().endswith("]")
          else (x if isinstance(x, list) else [x]))
)

# 'reference' 컬럼 코드는 그대로 둡니다.
if "reference" in dataset.columns:
    dataset["reference"] = dataset["reference"].apply(
        lambda x: "" if x is None or (isinstance(x, float) and pd.isna(x))
        else (x if isinstance(x, str) else "\n\n".join(map(str, x)))
    )

In [6]:
dataset.to_csv('response_data.csv')

In [7]:
# ===== RAGAS 평가: Ollama + SentenceTransformer (LangChain 없음) =====
import asyncio
from dataclasses import dataclass
from typing import Any, List, Optional

import ollama
import pandas as pd
from datasets import Dataset
from sentence_transformers import SentenceTransformer

from ragas import evaluate
from ragas.run_config import RunConfig
from ragas.llms import BaseRagasLLM
from ragas.embeddings import BaseRagasEmbeddings
from ragas.metrics import (
    context_precision,
    context_recall,
    faithfulness,
    answer_relevancy,
)

# -------------------------------------------------------------------
# 0) 컨텍스트 길이 줄이기 (토큰 수 줄여서 Timeout 완화)
# -------------------------------------------------------------------
def truncate_contexts(ctx_list, max_items=3, max_chars=1500):
    if ctx_list is None:
        return []
    if not isinstance(ctx_list, list):
        ctx_list = [ctx_list]

    trimmed = []
    for c in ctx_list[:max_items]:
        s = str(c)
        if len(s) > max_chars:
            s = s[:max_chars]
        trimmed.append(s)
    return trimmed

eval_df = dataset.copy()
eval_df["retrieved_contexts"] = eval_df["retrieved_contexts"].apply(truncate_contexts)
eval_df = eval_df.fillna({"response": "", "reference": ""})

# ragas 쪽에서 많이 쓰는 기본 컬럼 이름으로 맞추기
ragas_df = pd.DataFrame({
    "question": eval_df["user_input"],
    "contexts": eval_df["retrieved_contexts"],
    "answer": eval_df["response"],
    "ground_truth": eval_df["reference"],
})

hf_ds = Dataset.from_pandas(ragas_df)

# -------------------------------------------------------------------
# 1) RAGAS가 기대하는 LLM 결과 포맷 (간단 구조체)
# -------------------------------------------------------------------
@dataclass
class SimpleGeneration:
    text: str

@dataclass
class SimpleLLMResult:
    # ragas 내부에서 result.generations[0][0].text 이런 식으로 접근
    generations: List[List[SimpleGeneration]]


# -------------------------------------------------------------------
# 2) Ollama용 RAGAS LLM 래퍼
# -------------------------------------------------------------------
class OllamaRagasLLM(BaseRagasLLM):
    def __init__(
        self,
        model: str = "phi3:medium",
        host: str = "http://127.0.0.1:11434",
    ):
        super().__init__()
        # 여기 model은 "모델 이름" 문자열
        self.model = model
        self.client = ollama.Client(host=host)

    def _to_text(self, prompt: Any) -> str:
        if hasattr(prompt, "to_string"):
            return prompt.to_string()
        if hasattr(prompt, "text"):
            return prompt.text
        return str(prompt)

    def generate_text(
        self,
        prompt: Any,
        n: int = 1,
        temperature: float = 1e-8,
        stop: Optional[List[str]] = None,
        callbacks: Optional[Any] = None,
    ) -> SimpleLLMResult:
        prompt_text = self._to_text(prompt)
        messages = [{"role": "user", "content": prompt_text}]

        # 한 프롬프트에 대한 n개 생성 → [ [g1, g2, g3] ] 구조
        gens_for_single_prompt: List[SimpleGeneration] = []

        for _ in range(max(n, 1)):
            resp = self.client.chat(
                model=self.model,
                messages=messages,
                options={
                    "temperature": max(temperature, 0.0),
                },
            )
            text = resp["message"]["content"]

            if stop:
                for s in stop:
                    idx = text.find(s)
                    if idx != -1:
                        text = text[:idx]
                        break

            gens_for_single_prompt.append(SimpleGeneration(text=text))

        # 바깥 리스트: 프롬프트 하나, 안쪽 리스트: 그 프롬프트에 대한 여러 생성
        return SimpleLLMResult(generations=[gens_for_single_prompt])

    async def agenerate_text(
        self,
        prompt: Any,
        n: int = 1,
        temperature: float = 1e-8,
        stop: Optional[List[str]] = None,
        callbacks: Optional[Any] = None,
    ) -> SimpleLLMResult:
        loop = asyncio.get_event_loop()
        return await loop.run_in_executor(
            None,
            self.generate_text,
            prompt,
            n,
            temperature,
            stop,
            callbacks,
        )

    def is_finished(self, response: SimpleLLMResult) -> bool:
        try:
            for gen_list in response.generations:
                for gen in gen_list:
                    if not getattr(gen, "text", "").strip():
                        return False
            return True
        except Exception:
            return False


# -------------------------------------------------------------------
# 3) SentenceTransformer 기반 로컬 임베딩 (BaseRagasEmbeddings 구현)
# -------------------------------------------------------------------
class LocalHFEmbeddings(BaseRagasEmbeddings):
    def __init__(self, model_name: str = "intfloat/multilingual-e5-large-instruct"):
        super().__init__()
        # ragas 내부 로깅에서 embeddings.model 을 문자열로 기대하므로 이렇게
        self.model = model_name
        # 실제 HF 모델은 별도 속성에
        self._model = SentenceTransformer(model_name)

    # 동기 버전
    def embed_query(self, text: str) -> List[float]:
        emb = self._model.encode([text], normalize_embeddings=True)[0]
        return emb.tolist()

    def embed_documents(self, texts: List[str]) -> List[List[float]]:
        if len(texts) == 0:
            return []
        embs = self._model.encode(texts, normalize_embeddings=True)
        return [e.tolist() for e in embs]

    # 비동기 버전
    async def aembed_query(self, text: str) -> List[float]:
        return self.embed_query(text)

    async def aembed_documents(self, texts: List[str]) -> List[List[float]]:
        return self.embed_documents(texts)


# -------------------------------------------------------------------
# 4) 인스턴스 생성
# -------------------------------------------------------------------
# 평가용 LLM: 기본은 8B, 꼭 20B로 평가까지 하고 싶으면 model="gpt-oss:20b" 로 바꾸면 됨
llm = OllamaRagasLLM(
    model="gpt-oss:20b",
    host="http://127.0.0.1:11434",
)

embeddings = LocalHFEmbeddings(
    model_name="intfloat/multilingual-e5-large-instruct",
)

# -------------------------------------------------------------------
# 5) 메트릭 정의
# -------------------------------------------------------------------
metrics = [context_precision, context_recall, faithfulness, answer_relevancy]

# -------------------------------------------------------------------
# 6) run_config: 동시 작업 수 / 타임아웃 조절
# -------------------------------------------------------------------
run_config = RunConfig(
    timeout=1200,
    max_workers = 12  
)

# -------------------------------------------------------------------
# 7) 평가 실행
# -------------------------------------------------------------------
result = evaluate(
    dataset=hf_ds,
    metrics=metrics,
    llm=llm,
    embeddings=embeddings,
    run_config=run_config,
)

print(result)
result_df = result.to_pandas()
result_df.head()
# result_df.to_csv("ragas_eval_result.csv", index=False)

Evaluating: 100%|██████████| 120/120 [33:22<00:00, 16.69s/it]


{'context_precision': 0.7000, 'context_recall': 0.5167, 'faithfulness': 0.5914, 'answer_relevancy': 0.7006}


,user_input,retrieved_contexts,response,reference,context_precision,context_recall,faithfulness,answer_relevancy
0,(주)세토피아의 현재 대표이사는 누구인가?,[【 대표이사 등의 확인 】 전자공시시스템 dart.fss.or.kr Page 3반...,서상철,(주)세토피아의 현재 대표이사는 서상철입니다.,1.0,1.0,1.0,0.840014
1,(주)세토피아의 2022년 8월 1일 이전 상호명은 무엇이었나?,[2022년 8월 1일 임시주주총회 이후 (주)마이더스AI에서 (주)세토피아로 상호...,(주)마이더스AI,(주)마이더스AI였습니다.,1.0,1.0,1.0,0.839993
2,(주)세토피아가 (주)제이슨앤컴퍼니를 흡수합병한 합병기일은 언제인가?,[합병기일 2023.01.02],2023.01.02,2023년 1월 2일입니다.,1.0,1.0,0.0,0.778552
3,세토피아 철강사업의 주력 제품과 국내 시장 점유율은 어떻게 되는가?,"[<철강사업> 당사는 철강, 스테인레스강, 특수강 등의 도매, 제조사업을 영위하고 ...",제공된 컨텍스트만으로는 질문에 답변할 수 없습니다.,"주력 제품은 STS 201이며, 국내 시장점유율 약 30~40%로 1위를 차지하고 ...",0.0,0.0,0.0,0.000000
4,세토피아가 진출했던 유통사업은 어떤 브랜드 제품의 국내 총판 독점 계약이었나?,[2월 글로벌 1위 전자담배브랜드 ELFBAR(엘프바) 제품 국내 총판 독점 계약을...,ELFBAR(엘프바),글로벌 1위 전자담배브랜드 'ELFBAR(엘프바)' 제품의 국내 총판 독점 계약이었...,1.0,1.0,1.0,0.805494


In [10]:
result

{'context_precision': 0.7000, 'context_recall': 0.5167, 'faithfulness': 0.5914, 'answer_relevancy': 0.7006}

In [8]:
result_df

,user_input,retrieved_contexts,response,reference,context_precision,context_recall,faithfulness,answer_relevancy
0,(주)세토피아의 현재 대표이사는 누구인가?,[【 대표이사 등의 확인 】 전자공시시스템 dart.fss.or.kr Page 3반...,서상철,(주)세토피아의 현재 대표이사는 서상철입니다.,1.0,1.0,1.000000,0.840014
1,(주)세토피아의 2022년 8월 1일 이전 상호명은 무엇이었나?,[2022년 8월 1일 임시주주총회 이후 (주)마이더스AI에서 (주)세토피아로 상호...,(주)마이더스AI,(주)마이더스AI였습니다.,1.0,1.0,1.000000,0.839993
2,(주)세토피아가 (주)제이슨앤컴퍼니를 흡수합병한 합병기일은 언제인가?,[합병기일 2023.01.02],2023.01.02,2023년 1월 2일입니다.,1.0,1.0,0.000000,0.778552
3,세토피아 철강사업의 주력 제품과 국내 시장 점유율은 어떻게 되는가?,"[<철강사업> 당사는 철강, 스테인레스강, 특수강 등의 도매, 제조사업을 영위하고 ...",제공된 컨텍스트만으로는 질문에 답변할 수 없습니다.,"주력 제품은 STS 201이며, 국내 시장점유율 약 30~40%로 1위를 차지하고 ...",0.0,0.0,0.000000,0.000000
4,세토피아가 진출했던 유통사업은 어떤 브랜드 제품의 국내 총판 독점 계약이었나?,[2월 글로벌 1위 전자담배브랜드 ELFBAR(엘프바) 제품 국내 총판 독점 계약을...,ELFBAR(엘프바),글로벌 1위 전자담배브랜드 'ELFBAR(엘프바)' 제품의 국내 총판 독점 계약이었...,1.0,1.0,1.000000,0.805494
5,"제11기(당반기) 감사보고서의 감사의견은 무엇이며, 감사인은 누구인가?",[V. 회계감사인의 감사의견 등 1. 외부감사에 관한 사항 가. 회계감사인의 명칭 ...,"제11기(당반기) 감사보고서의 감사의견은 의견거절이며, 감사인은 삼일회계법인입니다.","감사인은 삼일회계법인이며, 감사의견은 '의견거절'입니다.",1.0,1.0,1.000000,0.873377
6,2025년 1월 20일 (주)에스에이코퍼레이션으로부터의 유형자산(토지 및 건물) 양...,[※ 관련공시 : 2025.01.20 주요사항보고서(유형자산양수결정) 주2) 202...,2025년 1월 20일 (주)에스에이코퍼레이션으로부터의 유형자산(토지 및 건물) 양...,2025년 1월 20일부로 해당 계약을 해지(철회)하였습니다.,1.0,1.0,0.000000,0.852979
7,2025년 2월 12일에 완료된 무상감자의 비율은 어떻게 되는가?,[ 합의의 건 - - 가결 6 2025.01.24 기타 경영 현황의 건(대여금 합...,제공된 컨텍스트만으로는 질문에 답변할 수 없습니다.,5:1 무상감자를 완료하였습니다.,0.0,0.0,0.000000,0.783260
8,"2025년 6월 30일 기준, 세토피아의 최대주주는 누구이며 지분율은 몇 %인가?",[세토피아 의 최대주주 변경은 없습니다. 4. 합병비율 (주)세토피아: (주)제이슨...,"2025년 6월 30일 기준, 세토피아의 최대주주는 (주)에스에이코퍼레이션으로 지분...","최대주주는 (주)에스에이코퍼레이션이며, 지분율은 6.89%입니다.",0.0,0.0,0.000000,0.925872
9,"2025년 3월 12일 증권선물위원회가 세토피아에 부과한 과징금 금액은 얼마이며, ...",[(*2) 2025년 03월 12일 증권선물위원회는 주식회사의 외부감사에 관한 법률...,"2025년 3월 12일 증권선물위원회가 세토피아에 부과한 과징금은 2.7억원이며, ...","과징금 2.7억원이 부과되었으며, 주된 사유는 금융자산·부채 과대계상(2019년 8...",1.0,1.0,0.500000,0.951304


In [9]:
result_df.to_csv('result.csv')